# Julian day & Local Sidereal Time (Algorithm 5.3)

**Goal:** turn a calendar date + Universal Time + a site's longitude into the **local sidereal time** (LST) — the angle, measured from the vernal equinox, of the site's meridian in the inertial frame. LST is what lets you place a ground observation into the geocentric-equatorial frame where orbits live.

**Code (answer key):** [`J0`](../curtis/algorithms/julian_day.py), [`LST`](../curtis/algorithms/alg_5_3_lst.py) · **Book:** §5.4–5.5, Algorithm 5.3 · **Worked examples:** 5.4 (J0) and 5.6 (LST)

No orbital dynamics here — just timekeeping. But it is the hinge between the *rotating* Earth and the *fixed* frame your state vectors use.

## Read first

| Symbol | Meaning |
|---|---|
| `J0` | Julian day number at 0 h UT (a continuous day count) |
| `j` | Julian centuries since the J2000 epoch |
| `g0` | Greenwich sidereal time at 0 h UT (deg) |
| `gst` | Greenwich sidereal time at the given UT (deg) |
| `lst` | local sidereal time = `gst` + east longitude (deg) |
| inputs | year, month, day, `ut` (hours), `EL` (east longitude, deg) |

**Julian day (Eq 5.48):** $J_0 = 367y - \big\lfloor\tfrac{7(y+\lfloor(m+9)/12\rfloor)}{4}\big\rfloor + \big\lfloor\tfrac{275m}{9}\big\rfloor + d + 1{,}721{,}013.5$ (here $\lfloor\cdot\rfloor$ truncates toward zero).

## The picture (why sidereal, not solar)

Your orbit lives in the inertial geocentric-equatorial frame, whose $\hat{\mathbf{I}}$ axis points at the **vernal equinox** — a fixed star direction. But a ground station is bolted to the spinning Earth. To use an observation, you need the station's orientation *in that fixed frame* at the moment of the observation. That angle is the **local sidereal time**.

- **Greenwich sidereal time (GST)** is the angle from $\hat{\mathbf{I}}$ to the Greenwich meridian — it grows as the Earth turns.
- **Local sidereal time** is just GST plus your **east longitude**: $\text{LST}=\text{GST}+\text{EL}$.

"Sidereal" (star-based), not "solar": the Earth spins $360.98564724°$ per day relative to the stars, not $360°$ — the extra $\approx0.9856°$ is one full turn spread over the year of orbital motion. That is why a sidereal day is ~3 m 56 s shorter than a solar day.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

In [ ]:
# Example 5.6 geometry, viewed down the Earth's north pole.
year, month, day, ut = 2004, 3, 3, 4.5
EL = 139 + 47/60                        # east longitude 139 deg 47' = 139.7833 deg

# (computed inline here; you will write J0 / LST below)
j0 = 367*year - int(7*(year + int((month+9)/12))/4) + int(275*month/9) + day + 1721013.5
j  = (j0 - 2451545)/36525
g0 = 100.4606184 + 36000.77004*j + 0.000387933*j**2 - 2.583e-8*j**3
g0 = g0 - int(g0/360)*360
gst = (g0 + 360.98564724*ut/24) % 360
lst = (gst + EL) % 360

def ray(angle_deg, color, label, r=1.0, lw=2):
    a = np.radians(angle_deg)
    ax.plot([0, r*np.cos(a)], [0, r*np.sin(a)], color=color, lw=lw)
    ax.annotate(label, (1.07*np.cos(a), 1.07*np.sin(a)), color=color,
                ha='center', va='center', fontsize=11)

fig, ax = plt.subplots(figsize=(6.2, 6.2))
circ = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(circ), np.sin(circ), color='#c2c0b6', lw=1)          # equator
ax.plot(0, 0, 'o', color='#3B8BD4', ms=10)                          # Earth (N pole)
ray(0,   '#5F5E5A', 'vernal equinox (Î)')
ray(gst, '#378ADD', f'Greenwich\n(GST={gst:.1f}°)')
ray(lst, '#D4537E', f'station\n(LST={lst:.1f}°)')
ax.add_patch(Arc((0,0), 0.7, 0.7, theta1=0, theta2=gst, color='#378ADD', lw=2))
ax.add_patch(Arc((0,0), 0.5, 0.5, theta1=gst, theta2=gst+EL, color='#1D9E75', lw=2))
ax.annotate(f'EL={EL:.1f}°', (0.34*np.cos(np.radians(gst+EL/2)),
            0.34*np.sin(np.radians(gst+EL/2))), color='#1D9E75', ha='center', fontsize=10)
ax.set_aspect('equal'); ax.axis('off'); ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
ax.set_title('Local sidereal time = Greenwich sidereal time + east longitude')
plt.show()

## See it — LST is GST plus your longitude

Looking down on the equator from the north pole: $\hat{\mathbf{I}}$ (vernal equinox) is the fixed reference at $0°$. The Greenwich meridian has swung to GST; your station sits a further **east longitude** around, at $\text{LST}=\text{GST}+\text{EL}$ (reduced to $0$–$360°$). As the Earth turns, both blue and pink rays sweep around together while $\hat{\mathbf{I}}$ stays put — and LST is the number you read off for the station at the observation instant.

## Where these equations come from

### `J0` — a single continuous day count (Eq 5.48)
Calendars are awkward (months of different lengths, leap years), so astronomers count **Julian days**: a running tally of days since 4713 BC. The integer-truncation formula bakes in the leap-year and month-length rules, returning the count at 0 h UT. Because it is one monotonic number, time differences become simple subtraction. (Add `ut/24` to get the JD at a non-midnight time, as in Example 5.4.)

### `j` — centuries since J2000 (Eq 5.49)
$j=(J_0-2451545)/36525$. The constant $2451545$ is the JD of the **J2000** epoch (2000 Jan 1, 12 h); $36525$ is days per Julian century. So $j$ measures time in centuries from J2000 — the variable the next polynomial expects.

### `g0` — Greenwich sidereal time at 0 h UT (Eq 5.50)
$g0 = 100.4606184 + 36000.77004\,j + 0.000387933\,j^2 - 2.583\times10^{-8} j^3$ (degrees). This empirical cubic is the IAU model of how far the Earth has rotated (relative to the stars) at midnight, fitted to observations. Reduce it to $0$–$360°$.

### `gst`, `lst` (Eqs 5.51, 5.52)
Add the rotation accumulated during the day, $360.98564724\,\tfrac{\text{ut}}{24}$ — note the rate is just over $360°$/day (the sidereal rate). Then $\text{lst}=\text{gst}+\text{EL}$, reduced to $0$–$360°$.

## Step by step (in code order)

**`J0(year, month, day)`** — Eq 5.48 directly (use `int(...)` for the truncations, since the arguments are positive).

**`LST(year, month, day, ut, EL)`:**
1. `j0 = J0(year, month, day)`
2. `j  = (j0 - 2451545)/36525`
3. `g0 = 100.4606184 + 36000.77004*j + 0.000387933*j**2 - 2.583e-8*j**3`
4. `g0 = zeroTo360(g0)`  (reduce to 0–360°)
5. `gst = g0 + 360.98564724*ut/24`
6. `lst = gst + EL`
7. `lst = lst - 360*int(lst/360)`  (reduce to 0–360°)

**↓ Now type the three functions below.** `J0` is a one-line transcription; `zeroTo360` reduces an angle to 0–360°; `LST` chains the steps. Answer key linked above; peek only after you try.

In [ ]:
def J0(year, month, day):
    """Julian day number at 0 h UT (Eq 5.48)."""
    # return 367*year - int(7*(year + int((month+9)/12))/4) + int(275*month/9) + day + 1721013.5
    raise NotImplementedError("transcribe Eq 5.48, then delete this line")


def zeroTo360(x):
    """Reduce an angle (deg) to the range 0-360."""
    # if x >= 360:  x -= int(x/360)*360
    # elif x < 0:   x -= (int(x/360) - 1)*360
    # return x
    raise NotImplementedError("fill in, then delete this line")


def LST(year, month, day, ut, EL):
    """Local sidereal time (deg) for a site at east longitude EL."""
    # 1. j0 = J0(year, month, day)
    # 2. j  = (j0 - 2451545)/36525
    # 3. g0 = 100.4606184 + 36000.77004*j + 0.000387933*j**2 - 2.583e-8*j**3
    # 4. g0 = zeroTo360(g0)
    # 5. gst = g0 + 360.98564724*ut/24
    # 6. lst = gst + EL
    # 7. lst = lst - 360*int(lst/360)
    # return lst
    raise NotImplementedError("fill in steps 1-7, then delete this line")

## Verify — Examples 5.4 and 5.6

**Example 5.4 (Julian day):** 2004-05-12, 14:45:30 UT → $J_0=2453137.5$, and $\text{JD}=J_0+\text{ut}/24=2453138.115$.

**Example 5.6 (local sidereal time):** site at east longitude $139.783°$, 2004-03-03, UT $=4.5$ h → $\text{LST}=8.57688°$ ($=0.571792$ h).

Run once the functions are typed.

In [ ]:
# Example 5.4 -- Julian day
j0 = J0(2004, 5, 12)
ut = 14 + 45/60 + 30/3600
jd = j0 + ut/24
print(f"J0 = {j0:.11g}   (expect 2453137.5)")
print(f"JD = {jd:.11g}   (expect 2453138.115)")
assert abs(j0 - 2453137.5) < 1e-6
assert abs(jd - 2453138.1149) < 1e-3

# Example 5.6 -- local sidereal time (site at 139 deg 47' E)
EL = 139 + 47/60
lst = LST(2004, 3, 3, 4.5, EL)
print(f"\nLST = {lst:.6g} deg  ({lst/15:.6g} h)   (expect 8.57688 deg, 0.571792 h)")
assert abs(lst - 8.57688) < 1e-4
print("\nAll checks passed ✔")

## What this confirms

- **Julian day** turns messy calendars into one continuous number, so time differences are just subtraction.
- **Sidereal vs solar:** the Earth turns $360.98564724°$ per day against the stars; that extra ~$0.99°$/day is the year of orbital motion folded in.
- **LST = GST + east longitude** is the single angle that places a ground site into the inertial frame — the prerequisite for converting observations into geocentric position vectors.

**Next:** Algorithm 5.4 (`rv_from_observe`) — combine LST with a radar measurement (range, azimuth, elevation, and their rates) to build the full geocentric state vector of the object.